# FMR — Real-Model Pipeline (Instance A)

Runs the Faithful Medical Reasoning pipeline on **real** open medical VLMs and datasets on a Colab GPU, then pushes results back to the `master` branch automatically.

**Before running:** set two Colab secrets (🔑 left sidebar):
- `HF_TOKEN` — Hugging Face read token (MedGemma also needs its license accepted on HF).
- `GH_TOKEN` — GitHub PAT with `contents: read+write` on `fmr-thesis` (for the auto-push).

Then `Runtime → Run all`. Pick a GPU runtime (T4 is enough for 2–3B models).

In [ ]:
# 1. GPU check
import torch, subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:600])
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime -> Change runtime type -> GPU"
print("CUDA OK:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Tokens from Colab secrets
import os
try:
    from google.colab import userdata
    for k in ("HF_TOKEN", "GH_TOKEN"):
        try: os.environ[k] = userdata.get(k)
        except Exception: print(f"WARNING: secret {k} not set")
except Exception:
    print("Not in Colab; expecting HF_TOKEN/GH_TOKEN already in env")
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("GH_TOKEN set:", bool(os.environ.get("GH_TOKEN")))

In [ ]:
# 3. Clone the repo (master = Instance A) and install
import os
REPO = "https://github.com/Ankit-blip737/fmr-thesis.git"
if not os.path.exists("fmr-thesis"):
    tok = os.environ.get("GH_TOKEN")
    url = REPO.replace("https://", f"https://x-access-token:{tok}@") if tok else REPO
    !git clone -b master {url} fmr-thesis
%cd fmr-thesis
!git pull --no-edit
!pip -q install -e fmr[real] || pip -q install numpy scipy pyyaml matplotlib scikit-learn transformers datasets accelerate qwen-vl-utils pillow
print("installed")

## 4. Run the pipeline

`run_real.py` runs baselines → blind test → full FMR (signals, FS, conformal gate) and writes to `fmr/outputs/real/<dataset>/`, then commits + pushes with `--push`.

Start with SLAKE (has real modality labels + closed subset). Add `pathvqa`, `vqa_rad` as time permits. `medvlm_r1` is the reasoning model; `qwen25_vl_3b` the non-reasoning baseline.

In [ ]:
# SLAKE — reasoning (MedVLM-R1) vs non-reasoning (Qwen2.5-VL-3B)
!cd fmr-thesis 2>/dev/null; python fmr/scripts/run_real.py     --dataset slake --reasoning medvlm_r1 --non-reasoning qwen25_vl_3b     --max-samples 400 --n-consistency 5 --alpha 0.10 --push

In [ ]:
# VQA-RAD (X-ray/CT). Small test set -> relax alpha for a feasible gate.
!python fmr/scripts/run_real.py     --dataset vqa_rad --reasoning medvlm_r1 --non-reasoning qwen25_vl_3b     --max-samples 400 --n-consistency 5 --alpha 0.15 --push

In [ ]:
# PathVQA (pathology modality breadth). Larger download.
!python fmr/scripts/run_real.py     --dataset pathvqa --reasoning medvlm_r1 --non-reasoning qwen25_vl_3b     --max-samples 400 --n-consistency 5 --alpha 0.15 --push

## 5. Inspect results
Figures + JSON are under `fmr/outputs/real/<dataset>/` and already pushed to `master`. On next local resume, Instance A pulls and wires these real FS scores into calibration (replacing the provisional mock scores).

In [ ]:
import json, glob
from IPython.display import Image, display
for f in sorted(glob.glob("fmr/outputs/real/*/fmr_results.json")):
    r = json.load(open(f))
    print(f"
=== {f} ===")
    print("  per-modality:", r.get("per_modality"))
    v = r.get("validation", {})
    print("  AUROC:", {k: round(v[k],3) for k in v if k.startswith("auroc_")})
    ab = r["abstention"]
    for g in ("fs","confidence"):
        print(f"  abstain[{g}] cov={ab[g]['test']['coverage']:.2f} err={ab[g]['test']['retained_error']} AURC={ab[g]['aurc']:.3f}")
for f in sorted(glob.glob("fmr/outputs/real/*/figures/fig1_grounding_drift.png")):
    display(Image(f))